In [ ]:
import os
from pyspark.sql import SparkSession

def get_spark_instance_via_adc(catalog_name: str, gcs_bucket: str) -> SparkSession:
    """
    Creates a Spark session configured for reading and writing Iceberg tables on GCS with ADC (Application Default Credentials).

    Args:
        catalog_name (str): The Iceberg catalog name. You can call it whatever you want
        gcs_bucket (str): The GCS bucket we are targeting for our iceberg warehouse

    Returns:
        SparkSession: Configured Spark session instance.
    """
    # Iceberg Runtime stuff
    spark_version = os.getenv("SPARK_VERSION", "3.5")
    scala_version = os.getenv("SCALA_VERSION", "2.12")
    iceberg_version = os.getenv("ICEBERG_VERSION", "1.7.0")

    iceberg_package = f"org.apache.iceberg:iceberg-spark-runtime-{spark_version}_{scala_version}:{iceberg_version}"

    # Define the Iceberg warehouse path
    warehouse_path = f"gs://{gcs_bucket}/icehouse"

    # this jar here: https://github.com/GoogleCloudDataproc/hadoop-connectors/releases
    local_jar_path = "./jars/gcs-connector-3.0.4-shaded.jar"

    return SparkSession.builder \
        .appName("local_spark_gcs") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config(f"spark.sql.catalog.{catalog_name}", "org.apache.iceberg.spark.SparkCatalog") \
        .config(f"spark.sql.catalog.{catalog_name}.type", "hadoop") \
        .config(f"spark.sql.catalog.{catalog_name}.warehouse", warehouse_path) \
        .config("spark.jars.packages", iceberg_package) \
        .config("spark.jars", local_jar_path) \
        .config("spark.hadoop.google.cloud.auth.service.account.enable", "false") \
        .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
        .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
        .config("spark.driver.host", "localhost") \
        .config("spark.driver.bindAddress", "127.0.0.1") \
        .config("spark.hadoop.google.cloud.auth.type", "APPLICATION_DEFAULT") \
        .getOrCreate()